# Demo 02: Semantic Tool Selection

A small model with a big toolbox gets confused: it picks the wrong tool, or none. Filtering the toolbox to the few tools relevant to each question — with a hook, not Python around the agent — improves tool-selection accuracy AND cuts tokens. Details in the [README](./README.md).

**Prerequisites:** AWS credentials (Amazon Bedrock), a running Neo4j with the hotel graph (demo 01's `build_graph.py`).

In [ ]:
# !pip install -q -r requirements.txt   # self-paced only; pre-installed at AWS events
import os
os.environ.setdefault("OTEL_SDK_DISABLED", "true")
os.environ.setdefault("AWS_REGION", "us-east-1")
from dotenv import load_dotenv
load_dotenv()

## Part 1: Agent without semantic tool search

`ALL_TOOLS` is a ~40-tool travel pool (real hotel/weather/booking tools + realistic stand-ins) with lots of overlap — many `get_*`, `search_*`, `check_*` variants. We use **Claude Haiku** (small) so the confusion is visible.

To score the demo we need to know which tool the agent actually called. `ToolTracker` is a tiny hook that just records the name of each tool call — it is measurement only, not part of the semantic-selection solution. The real harness (the `SemanticToolHook`) comes in Part 2.

In [ ]:
from strands import Agent
from strands.models import BedrockModel
from strands.hooks import BeforeToolCallEvent, HookProvider
from semantic_tools import ALL_TOOLS   # the ~40-tool travel pool (hotels, weather, bookings, + stand-ins)

# A small model so the confusion from a big toolset is visible.
MODEL = BedrockModel(model_id="global.anthropic.claude-haiku-4-5-20251001-v1:0",
                     region_name=os.environ["AWS_REGION"])

# Measurement-only hook: records the name of each tool the agent calls.
class ToolTracker(HookProvider):
    def __init__(self):
        self.called = []
    def register_hooks(self, registry):
        registry.add_callback(BeforeToolCallEvent, lambda e: self.called.append(e.tool_use["name"]))

print(f"Pool size: {len(ALL_TOOLS)} tools")

## Part 2: Agent with semantic tool search

The `SemanticToolHook` below runs on every request: it embeds the question, picks the top-3 most similar tools, and swaps the agent's tool registry to just those. The agent still owns all 40 tools; the hook narrows them per request.

In [ ]:
import json, boto3, faiss, numpy as np
from strands.hooks import BeforeInvocationEvent, HookRegistry

# The harness: on every request, embed the user's question, find the top-k tools
# by similarity over each tool's name+description, and replace the agent's
# tool_registry with just those. The agent is built with ALL tools; the hook
# trims them per request. This is a Strands HookProvider, not code around the agent.
class SemanticToolHook(HookProvider):
    def __init__(self, tools, top_k=3):
        self.tools, self.top_k, self.last_selected = tools, top_k, []
        self._bedrock = boto3.client("bedrock-runtime", region_name=os.environ["AWS_REGION"])
        vecs = np.array([self._embed(f"{t.tool_name}: {t.tool_spec['description']}") for t in tools], dtype="float32")
        self._index = faiss.IndexFlatL2(vecs.shape[1]); self._index.add(vecs)
    def _embed(self, text):
        r = self._bedrock.invoke_model(modelId="amazon.nova-2-multimodal-embeddings-v1:0",
            body=json.dumps({"taskType": "SINGLE_EMBEDDING", "singleEmbeddingParams": {
                "embeddingPurpose": "GENERIC_INDEX", "embeddingDimension": 1024,
                "text": {"truncationMode": "END", "value": text[:8000]}}}),
            contentType="application/json", accept="application/json")
        return json.loads(r["body"].read())["embeddings"][0]["embedding"]
    def register_hooks(self, registry: HookRegistry):
        registry.add_callback(BeforeInvocationEvent, self._trim)
    def _trim(self, event: BeforeInvocationEvent):
        msgs = event.messages or getattr(event.agent, "messages", [])
        query = next(("".join(b.get("text", "") for b in m.get("content", []))
                      for m in reversed(msgs) if m.get("role") == "user"), "")
        if not query:
            return
        _, idx = self._index.search(np.array([self._embed(query)], dtype="float32"), self.top_k)
        selected = [self.tools[i] for i in idx[0]]
        self.last_selected = [t.tool_name for t in selected]
        reg = event.agent.tool_registry
        reg.registry.clear(); reg.dynamic_tools.clear()
        for t in selected:
            reg.register_tool(t)

# Create the hook: it embeds all 40 tools once (name + description) into a FAISS
# index now, and will keep the top-3 (top_k) most relevant per request later.
hook = SemanticToolHook(ALL_TOOLS, top_k=3)

# Each question with the tool we expect the agent to pick (ground truth for scoring).
TESTS = [
    ("What is the weather forecast for Tokyo next week?", "get_weather_forecast"),
    ("How much does a room cost at Cliffside Resort?", "get_hotel_room_rates"),
    ("What amenities does Cliffside Resort have?", "get_hotel_amenities"),
    ("Find hotels in Barcelona", "search_hotels_by_city"),
    ("Book Cliffside Resort for John Smith for 2 nights", "book_hotel"),
    ("What is the current temperature in Paris?", "get_current_weather"),
    ("Is flight AA123 on time?", "get_flight_status"),
    ("Do I need a visa for Japan?", "get_visa_requirements"),
    ("What is the phone number for Cliffside Resort?", "get_hotel_contact"),
    ("Can I cancel my hotel reservation BK0001?", "cancel_booking"),
]
print("Hook ready")

## Part 3: Measure accuracy and tokens

Same questions, same small model. We score tool-selection **accuracy** and total **tokens** with and without the hook.

In [ ]:
# TEST A — all 40 tools every query (no filtering)
all_correct = all_in = all_out = 0
for q, expected in TESTS:
    tracker = ToolTracker()
    result = Agent(model=MODEL, system_prompt="You are a travel agent that helps people with hotels. Keep answers concise — at most two sentences.", tools=ALL_TOOLS, hooks=[tracker])(q)
    ok = expected in tracker.called
    u = result.metrics.accumulated_usage
    tin, tout = u["inputTokens"], u["outputTokens"]
    all_correct += ok; all_in += tin; all_out += tout
    print(f"  {'OK ' if ok else 'BAD'} want {expected:22} got {tracker.called}")
    print(f"        tokens: {tin}in / {tout}out")
print(f"\nALL 40 TOOLS   accuracy={all_correct}/{len(TESTS)}   tokens={all_in}in/{all_out}out")

In [ ]:
# TEST B — top-3 tools per query (the SemanticToolHook trims the pool; tracker scores it)
top3_correct = top3_in = top3_out = 0
for q, expected in TESTS:
    tracker = ToolTracker()
    result = Agent(model=MODEL, system_prompt="You are a travel agent that helps people with hotels. Keep answers concise — at most two sentences.", tools=ALL_TOOLS, hooks=[hook, tracker])(q)
    ok = expected in tracker.called
    u = result.metrics.accumulated_usage
    tin, tout = u["inputTokens"], u["outputTokens"]
    top3_correct += ok; top3_in += tin; top3_out += tout
    print(f"  {'OK ' if ok else 'BAD'} want {expected:22} got {tracker.called}")
    print(f"        tokens: {tin}in / {tout}out")
print(f"\nTOP-3 FILTERED accuracy={top3_correct}/{len(TESTS)}   tokens={top3_in}in/{top3_out}out")

In [ ]:
# RESULT — accuracy up, tokens down
print(f"Accuracy:  all={all_correct}/{len(TESTS)}  ->  top-3={top3_correct}/{len(TESTS)}")
print(f"Input tokens:  all={all_in}  ->  top-3={top3_in}  ({100*(all_in-top3_in)//all_in}% fewer)")
print(f"Output tokens: all={all_out}  ->  top-3={top3_out}")

Fewer, relevant tools = less confusion (higher accuracy) and far fewer tokens. The one trade-off: if FAISS misranks the right tool out of the top-3, the agent can't call it — the README discusses tuning `top_k` and tool descriptions.

## Part 4: Semantic tool search with memory

A **session manager** persists the agent's conversation to disk so it resumes after a restart. `FileSessionManager` writes to `./sessions/`; create an agent later with the same `session_id` and it remembers the conversation.

In [ ]:
from strands.session import FileSessionManager

session = FileSessionManager(session_id="demo02-memory", storage_dir="./sessions/")
agent = Agent(model=MODEL, system_prompt="You are a travel agent that helps people with hotels. Keep answers concise — at most two sentences.",
              tools=ALL_TOOLS, hooks=[hook], session_manager=session)

result = agent("Find hotels in Barcelona")
u = result.metrics.accumulated_usage
print(f"\n   Tokens: {u['inputTokens']}in/{u['outputTokens']}out")
print(f"   Conversation persisted to ./sessions/ (session_id=demo02-memory)")

## 🎯 Try It Yourself

In [ ]:
# Ask anything — the hook trims 40 tools to the 3 most relevant
q = "What's the exchange rate from USD to EUR?"
result = agent(q)
print(f"\n   Selected tools: {hook.last_selected}")
print(f"   Tokens: {result.metrics.accumulated_usage['totalTokens']}")

Interactive REPL: `python chat.py`. **Next:** [Demo 03 — Multi-Agent Validation](../03-multiagent-demo/).